In [1]:
import pandas as pd
import numpy as np

PATH = "malnutrition_children_ethiopia.csv"
df = pd.read_csv(PATH)

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

print("\nNutrition_Status distribution:")
print(df["Nutrition_Status"].value_counts(dropna=False))

df["target"] = (df["Nutrition_Status"] == "Malnourished").astype(int)

print("\nBinary target distribution (1=Malnourished):")
print(df["target"].value_counts())
print("\nPositive rate:", df["target"].mean())

na = df.isna().mean().sort_values(ascending=False)
print("\nMissing-value rate (top):")
print(na.head(10))

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("\nNumeric summary (selected):")
print(df[num_cols].describe().T[["min", "50%", "mean", "max"]])

if "ID" in df.columns:
    print("\nUnique IDs:", df["ID"].nunique(), "out of", len(df))


Shape: (4098, 16)

Columns: ['ID', 'Age (months)', 'Gender', 'Region', 'Mother_Education', 'Household_Wealth_Index', 'Height_cm', 'Weight_kg', 'Stunting', 'Underweight', 'Overweight', 'Anemia', 'Malaria', 'Diarrhea', 'TB', 'Nutrition_Status']

Nutrition_Status distribution:
Nutrition_Status
Normal          2030
At_Risk         1238
Malnourished     830
Name: count, dtype: int64

Binary target distribution (1=Malnourished):
target
0    3268
1     830
Name: count, dtype: int64

Positive rate: 0.2025378233284529

Missing-value rate (top):
ID                        0.0
Age (months)              0.0
Gender                    0.0
Region                    0.0
Mother_Education          0.0
Household_Wealth_Index    0.0
Height_cm                 0.0
Weight_kg                 0.0
Stunting                  0.0
Underweight               0.0
dtype: float64

Numeric summary (selected):
               min     50%         mean     max
ID             1.0  2049.5  2049.500000  4098.0
Age (months)   0.0

In [2]:
from sklearn.model_selection import train_test_split
PATH = "malnutrition_children_ethiopia.csv"
df = pd.read_csv(PATH)

df["target"] = (df["Nutrition_Status"] == "Malnourished").astype(int)

df = df.drop(columns=["ID", "Nutrition_Status"])

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,      
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,      
    stratify=y_temp,
    random_state=42
)

print("Split sizes:")
print("Train:", X_train.shape, "Pos rate:", y_train.mean())
print("Val:  ", X_val.shape,   "Pos rate:", y_val.mean())
print("Test: ", X_test.shape,  "Pos rate:", y_test.mean())

def add_features(split: pd.DataFrame) -> pd.DataFrame:
    split = split.copy()
    split["BMI"] = split["Weight_kg"] / (split["Height_cm"] / 100.0) ** 2

    split["Weight_per_Month"] = split["Weight_kg"] / (split["Age (months)"] + 1)
    split["Height_per_Month"] = split["Height_cm"] / (split["Age (months)"] + 1)

    age_bins = [0, 6, 12, 24, 36, 48, 60]
    age_labels = ["0-6", "6-12", "12-24", "24-36", "36-48", "48-60"]
    split["Age_Group"] = pd.cut(
        split["Age (months)"],
        bins=age_bins,
        labels=age_labels,
        include_lowest=True
    )

    disease_cols = ["Anemia", "Malaria", "Diarrhea", "TB"]
    split["Disease_Count"] = split[disease_cols].sum(axis=1)
    split["Has_Multiple_Diseases"] = (split["Disease_Count"] >= 2).astype(int)

    nutrition_flags = ["Stunting", "Underweight", "Overweight"]
    split["Nutrition_Risk_Score"] = split[nutrition_flags].sum(axis=1)

    edu_map = {"None": 0, "Primary": 1, "Secondary": 2, "Higher": 3}
    wealth_map = {"Poorest": 0, "Poor": 1, "Middle": 2, "Rich": 3, "Richest": 4}

    split["Mother_Education_Ord"] = split["Mother_Education"].map(edu_map)
    split["Wealth_Ord"] = split["Household_Wealth_Index"].map(wealth_map)

    split = split.drop(columns=["Mother_Education", "Household_Wealth_Index"])

    return split


X_train_fe = add_features(X_train)
X_val_fe   = add_features(X_val)
X_test_fe  = add_features(X_test)

cat_cols = ["Gender", "Region", "Age_Group"]

X_train_fe = pd.get_dummies(X_train_fe, columns=cat_cols, drop_first=True)
X_val_fe   = pd.get_dummies(X_val_fe,   columns=cat_cols, drop_first=True)
X_test_fe  = pd.get_dummies(X_test_fe,  columns=cat_cols, drop_first=True)

X_train_fe, X_val_fe = X_train_fe.align(X_val_fe, join="left", axis=1, fill_value=0)
X_train_fe, X_test_fe = X_train_fe.align(X_test_fe, join="left", axis=1, fill_value=0)

print("\nAfter feature engineering:")
print("Train:", X_train_fe.shape, "Val:", X_val_fe.shape, "Test:", X_test_fe.shape)

print("NaN check (train/val/test):",
      X_train_fe.isna().sum().sum(),
      X_val_fe.isna().sum().sum(),
      X_test_fe.isna().sum().sum())

print("\nSample engineered columns:", [c for c in X_train_fe.columns if c in
      ["BMI", "Weight_per_Month", "Height_per_Month", "Disease_Count",
       "Has_Multiple_Diseases", "Nutrition_Risk_Score", "Mother_Education_Ord", "Wealth_Ord"]])


Split sizes:
Train: (2868, 14) Pos rate: 0.2025801952580195
Val:   (615, 14) Pos rate: 0.2032520325203252
Test:  (615, 14) Pos rate: 0.2016260162601626

After feature engineering:
Train: (2868, 28) Val: (615, 28) Test: (615, 28)
NaN check (train/val/test): 2649 568 545

Sample engineered columns: ['BMI', 'Weight_per_Month', 'Height_per_Month', 'Disease_Count', 'Has_Multiple_Diseases', 'Nutrition_Risk_Score', 'Mother_Education_Ord', 'Wealth_Ord']


In [3]:
from sklearn.impute import SimpleImputer

for col in ["Mother_Education_Ord", "Wealth_Ord"]:
    for split in [X_train_fe, X_val_fe, X_test_fe]:
        split[col] = split[col].fillna(0)  # lowest category = safest assumption


num_cols = X_train_fe.select_dtypes(include=["int64", "float64"]).columns

num_imputer = SimpleImputer(strategy="median")

X_train_fe[num_cols] = num_imputer.fit_transform(X_train_fe[num_cols])
X_val_fe[num_cols]   = num_imputer.transform(X_val_fe[num_cols])
X_test_fe[num_cols]  = num_imputer.transform(X_test_fe[num_cols])

print("NaNs after fix:",
      X_train_fe.isna().sum().sum(),
      X_val_fe.isna().sum().sum(),
      X_test_fe.isna().sum().sum())


NaNs after fix: 0 0 0


In [4]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    classification_report
)

In [5]:
def evaluate_model(name, model, X_tr, y_tr, X_val, y_val):
    model.fit(X_tr, y_tr)

    val_probs = model.predict_proba(X_val)[:, 1]

    roc = roc_auc_score(y_val, val_probs)
    pr  = average_precision_score(y_val, val_probs)

    print(f"\n{name}")
    print("-" * len(name))
    print(f"ROC-AUC : {roc:.4f}")
    print(f"PR-AUC  : {pr:.4f}")

    return {
        "model": model,
        "roc_auc": roc,
        "pr_auc": pr,
        "val_probs": val_probs
    }

In [6]:
log_reg = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="lbfgs",
    n_jobs=-1
)

logreg_results = evaluate_model(
    "Logistic Regression",
    log_reg,
    X_train_fe, y_train,
    X_val_fe, y_val
)


Logistic Regression
-------------------
ROC-AUC : 0.4897
PR-AUC  : 0.1899


In [7]:
rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=12,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_results = evaluate_model(
    "Random Forest",
    rf,
    X_train_fe, y_train,
    X_val_fe, y_val
)


Random Forest
-------------
ROC-AUC : 0.5145
PR-AUC  : 0.2056


In [8]:
from catboost import CatBoostClassifier

cat = CatBoostClassifier(
    iterations=800,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    class_weights=[1, (len(y_train) - y_train.sum()) / y_train.sum()],
    verbose=0,
    random_seed=42
)

cat_results = evaluate_model(
    "CatBoost",
    cat,
    X_train_fe, y_train,
    X_val_fe, y_val
)


CatBoost
--------
ROC-AUC : 0.4864
PR-AUC  : 0.1940


In [9]:
from xgboost import XGBClassifier

pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

xgb = XGBClassifier(
    n_estimators=600,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_results = evaluate_model(
    "XGBoost",
    xgb,
    X_train_fe, y_train,
    X_val_fe, y_val
)


XGBoost
-------
ROC-AUC : 0.5173
PR-AUC  : 0.2013


In [10]:
results_df = pd.DataFrame([
    {"Model": "Logistic Regression", "PR-AUC": logreg_results["pr_auc"]},
    {"Model": "Random Forest",       "PR-AUC": rf_results["pr_auc"]},
    {"Model": "CatBoost",            "PR-AUC": cat_results["pr_auc"]},
    {"Model": "XGBoost",             "PR-AUC": xgb_results["pr_auc"]},
]).sort_values("PR-AUC", ascending=False)

print(results_df)

                 Model    PR-AUC
1        Random Forest  0.205635
3              XGBoost  0.201277
2             CatBoost  0.193976
0  Logistic Regression  0.189897


In [11]:
from sklearn.metrics import accuracy_score

accuracy_results = []

logreg_val_preds = logreg_results["model"].predict(X_val_fe)
accuracy_results.append({
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_val, logreg_val_preds)
})


rf_val_preds = rf_results["model"].predict(X_val_fe)
accuracy_results.append({
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_val, rf_val_preds)
})


cat_val_preds = cat_results["model"].predict(X_val_fe)
accuracy_results.append({
    "Model": "CatBoost",
    "Accuracy": accuracy_score(y_val, cat_val_preds)
})


xgb_val_preds = xgb_results["model"].predict(X_val_fe)
accuracy_results.append({
    "Model": "XGBoost",
    "Accuracy": accuracy_score(y_val, xgb_val_preds)
})

accuracy_df = pd.DataFrame(accuracy_results).sort_values("Accuracy", ascending=False)
print(accuracy_df)

                 Model  Accuracy
3              XGBoost  0.700813
2             CatBoost  0.700813
1        Random Forest  0.699187
0  Logistic Regression  0.515447


In [12]:
rf_val_probs  = rf_results["model"].predict_proba(X_val_fe)[:, 1]
xgb_val_probs = xgb_results["model"].predict_proba(X_val_fe)[:, 1]
cat_val_probs = cat_results["model"].predict_proba(X_val_fe)[:, 1]

In [14]:
#baseline Ensemble
ensemble_val_probs = (
    rf_val_probs +
    xgb_val_probs +
    cat_val_probs
) / 3.0

In [15]:
#validation
from sklearn.metrics import roc_auc_score, average_precision_score

print("Ensemble ROC-AUC:", roc_auc_score(y_val, ensemble_val_probs))
print("Ensemble PR-AUC :", average_precision_score(y_val, ensemble_val_probs))

Ensemble ROC-AUC: 0.5040816326530612
Ensemble PR-AUC : 0.19711207784251278


In [17]:
#weighted soft voting
ensemble_val_probs_weighted = (
    0.4 * rf_val_probs +
    0.3 * xgb_val_probs +
    0.3 * cat_val_probs
)

In [18]:
print("Weighted Ensemble ROC-AUC:",
      roc_auc_score(y_val, ensemble_val_probs_weighted))
print("Weighted Ensemble PR-AUC :",
      average_precision_score(y_val, ensemble_val_probs_weighted))

Weighted Ensemble ROC-AUC: 0.5044081632653061
Weighted Ensemble PR-AUC : 0.19716280992286728


In [23]:
#Thereshold Optimization
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(
    y_val, ensemble_val_probs_weighted
)

pr_df = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision[:-1],
    "recall": recall[:-1]
})

candidate = pr_df[
    (pr_df["recall"] >= 0.70) &
    (pr_df["precision"] >= 0.21)
].head(1)

candidate

,threshold,precision,recall
73,0.210204,0.210332,0.912


In [24]:
#final eval
rf_test_probs  = rf_results["model"].predict_proba(X_test_fe)[:, 1]
xgb_test_probs = xgb_results["model"].predict_proba(X_test_fe)[:, 1]
cat_test_probs = cat_results["model"].predict_proba(X_test_fe)[:, 1]

ensemble_test_probs = (
    0.4 * rf_test_probs +
    0.3 * xgb_test_probs +
    0.3 * cat_test_probs
)

best_threshold = float(candidate["threshold"].values[0])

ensemble_test_preds = (ensemble_test_probs >= best_threshold).astype(int)


In [25]:
#metrics
from sklearn.metrics import classification_report, confusion_matrix

print("TEST ROC-AUC:",
      roc_auc_score(y_test, ensemble_test_probs))
print("TEST PR-AUC :",
      average_precision_score(y_test, ensemble_test_probs))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, ensemble_test_preds))

print("\nClassification Report:")
print(classification_report(y_test, ensemble_test_preds, digits=3))

TEST ROC-AUC: 0.5049109782537284
TEST PR-AUC : 0.20515414120213876

Confusion Matrix:
[[ 80 411]
 [ 14 110]]

Classification Report:
              precision    recall  f1-score   support

           0      0.851     0.163     0.274       491
           1      0.211     0.887     0.341       124

    accuracy                          0.309       615
   macro avg      0.531     0.525     0.307       615
weighted avg      0.722     0.309     0.287       615



In [26]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Validation predictions
ensemble_val_preds = (ensemble_val_probs_weighted >= best_threshold).astype(int)

val_metrics = {
    "Accuracy": accuracy_score(y_val, ensemble_val_preds),
    "Precision": precision_score(y_val, ensemble_val_preds),
    "Recall": recall_score(y_val, ensemble_val_preds),
    "F1-score": f1_score(y_val, ensemble_val_preds)
}

print("Validation Metrics (Ensemble):")
for k, v in val_metrics.items():
    print(f"{k}: {v:.4f}")


Validation Metrics (Ensemble):
Accuracy: 0.2862
Precision: 0.2103
Recall: 0.9120
F1-score: 0.3418


In [27]:
# Test predictions
ensemble_test_preds = (ensemble_test_probs >= best_threshold).astype(int)

test_metrics = {
    "Accuracy": accuracy_score(y_test, ensemble_test_preds),
    "Precision": precision_score(y_test, ensemble_test_preds),
    "Recall": recall_score(y_test, ensemble_test_preds),
    "F1-score": f1_score(y_test, ensemble_test_preds)
}

print("\nTest Metrics (Ensemble):")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}")


Test Metrics (Ensemble):
Accuracy: 0.3089
Precision: 0.2111
Recall: 0.8871
F1-score: 0.3411


In [28]:
def get_metrics(y_true, probs, threshold):
    preds = (probs >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, preds),
        "Precision": precision_score(y_true, preds),
        "Recall": recall_score(y_true, preds),
        "F1-score": f1_score(y_true, preds)
    }

comparison = pd.DataFrame.from_dict({
    "Random Forest": get_metrics(y_val, rf_val_probs, best_threshold),
    "XGBoost":       get_metrics(y_val, xgb_val_probs, best_threshold),
    "CatBoost":      get_metrics(y_val, cat_val_probs, best_threshold),
    "Ensemble":      get_metrics(y_val, ensemble_val_probs_weighted, best_threshold)
}, orient="index")

print(comparison)

               Accuracy  Precision  Recall  F1-score
Random Forest  0.203252   0.203252   1.000  0.337838
XGBoost        0.497561   0.230994   0.632  0.338330
CatBoost       0.437398   0.202156   0.600  0.302419
Ensemble       0.286179   0.210332   0.912  0.341829
